### Figure 1 D + E & Figure 4 A + B + E

In [ ]:
#if (!require("BiocManager", quietly = TRUE))
#    install.packages("BiocManager")
#
#BiocManager::install("ComplexHeatmap")

In [ ]:
install.packages("Cairo")

In [ ]:
library(readxl)
library(tidyverse)
library(ggrepel)
library(ggtext)
library(ggExtra)
library(vegan)
library(glue)

In [ ]:
### load sample meta data
metadata <- read_excel("metadata_table_new.xlsx") %>%
  filter(study == "Lehmann" | study == "Henry" | study == "Thuy-Boun" | study == "Lloyd-Price") %>%
  # filter for comparision of all IBD patients vs healthy individuals
  filter(disease == "normal" | disease == "UC" | disease == "CD") %>%
  # filter for comparision of CD patients vs healthy individuals
  #filter(disease == "normal" | disease == "CD") %>%
  # filter for comparision of UC patients vs healthy individuals
  #filter(disease == "normal" | disease == "UC") %>%
  # filter for comparision of UC patients vs CD patients
  # filter(disease == "UC" | disease == "CD") %>%
  mutate(sample = min_rank(sample)) %>%
  mutate(batch = case_when(batch == 2 ~ 1, batch == 4 ~ 2, batch == 8 ~ 3, batch == 9 ~ 4, TRUE ~ batch)) %>%
  as.data.frame()


In [ ]:
# can be skiped for UC vs CD
metadata <- metadata[!metadata$ID == "P06b_C", ]
metadata <- metadata[!metadata$ID == "P06c_C", ]
metadata

In [ ]:
shared_metaproteins <- read.delim("shared_1620_after_batch(2).csv",check.names = FALSE, header = TRUE, sep = ",") %>%column_to_rownames(var = "GroupID")%>% select(-(1:2)) 
head((shared_metaproteins)) 

In [ ]:
#### filter columns to select only samples that are goal of this analysis

filter_meta <- shared_metaproteins %>% 
  t() %>%
  as.data.frame() %>%
  rownames_to_column(var = "ID") %>%
  inner_join(metadata, ., by = "ID") %>%
  select(-(2:8)) %>%
  column_to_rownames(var = "ID") %>%
  t() %>%
  as.data.frame() %>%
  filter(rowSums(.) != 0)
filter_meta


In [ ]:
############################### old: 
### METAPROTEINS ###
### Input data: metaproteins identified in all studies before / after batch effect correction


######### new
rel_abund <-filter_meta
# variance analysis for IBD and study

feat.conf <- rel_abund
label.conf <- metadata$disease # metadata$condition or metadata$disease --> control / diseased --> for UC vs CD, this has to be changend to "disease" and UC/UCr/UCa have to be united
names(label.conf) <- colnames(feat.conf)
study <- as.factor(metadata$study)
names(study) <- colnames(feat.conf)

var.label <- vapply(rownames(feat.conf), FUN = function(x) {
  x <- feat.conf[x, ]
  x <- rank(x)/length(x)
  ss.tot <- sum((x - mean(x, na.rm = TRUE))^2)/length(x)
  ss.o.i <- sum(vapply(unique(label.conf), function(s) {
    sum((x[label.conf == s] - mean(x[label.conf == s], na.rm = TRUE))^2)
  }, FUN.VALUE = double(1)))/length(x)
  return(1 - ss.o.i/ss.tot)
}, FUN.VALUE = double(1))

if (any(is.infinite(var.label))) {
  var.label[is.infinite(var.label)] <- NA
}

var.batch <- vapply(rownames(feat.conf), FUN = function(x) {
  x <- feat.conf[x, names(study)]
  x <- rank(x)/length(x)
  ss.tot <- sum((x - mean(x, na.rm = TRUE))^2)/length(x)
  ss.o.i <- sum(vapply(levels(study), function(s) {
    sum((x[study == s] - mean(x[study == s], na.rm = TRUE))^2)
  }, FUN.VALUE = double(1)))/length(x)
  return(1 - ss.o.i/ss.tot)
}, FUN.VALUE = double(1))

if (any(is.infinite(var.batch))) {
  var.batch[is.infinite(var.batch)] <- NA
}


df.plot <- tibble(label = var.label, batch = var.batch, species=names(var.label))

df.plot$mean <- rowMeans(feat.conf * 100, na.rm = TRUE)

df.plot %>% 
  ggplot(aes(x = label, y = batch, size = mean)) + 
  
  geom_point(data = subset(df.plot, label >= 0.2 & label > batch),
             aes(x = label, y = batch, size = mean),
             shape = 21, show.legend = FALSE, fill = alpha("red3", 0.8), color = "white", stroke = 0.2) +
  
  geom_point(data = subset(df.plot, label < 0.2 | label < batch), 
             aes(x = label, y = batch, size = mean), 
             shape = 21, show.legend = TRUE, fill = alpha("grey10", 0.4), color = "white", stroke = 0.2) +
  
  labs(y = "Variance explained by Study", x = "Variance explained by IBD", size = "Mean rel. ab.") +
  theme_bw() +
  theme(aspect.ratio = 1, 
        legend.position = "right",  
        legend.direction = "vertical",
        legend.box = "vertical",
        legend.justification = "bottom",
        axis.text = element_text(color = "black"),
        legend.text = element_text(size = 6),
        legend.title = element_text(size = 8),
        panel.grid = element_blank() ) + 
  scale_size_continuous(range = c(1, 8), labels = scales::percent_format(scale = 1)) + 
  xlim(0, 0.8) + 
  ylim(0, 0.8) + 
  geom_abline(intercept = 0, slope = 1, col = 'grey40', linetype = 3)

  ## save plot
  ggsave("variance_explained_shared_UCvsCD.png", width = 13, height = 13, units = "cm", dpi = 1000)


In [ ]:
# filter for proteins that have a higher variance explained by IBD than study and a variance explained by study <= 0.20 
filtered_df.plot <- df.plot %>%
  filter(label >= 0.20, label > batch) %>%
  select(1:4)
# save these potential biomarkers for further analysis
write.csv(filtered_df.plot, file = "UCvsCD.csv", row.names = FALSE)